# பாடம் 01 - AI முகவர்கள் அறிமுகம்

**துவக்க நிலை AI முகவர்கள்** பாடநெறியின் முதல் பாடத்திற்கு வரவேற்பு!

**AI முகவர்** என்பது ஒரு பெரிய மொழி மாதிரியை (LLM) அதன் காரணப்படுத்தும் இயந்திரமாக பயன்படுத்தும் ஒரு நிகழ்ச்சி ஆகும் மற்றும் அது உலகில் *செயல்கள்* செய்ய முடியும் — APIகளை அழைப்பு செய்தல், தரவுத்தளங்களை விசாரித்தல், அல்லது குறியீட்டை இயக்குதல் — ஒரு பயனருக்காக ஒரு இலக்கை அடைவதற்கு.

இந்த நோட்புக்கில் நீங்கள் உங்கள் முதல் முகவரான **பயண முகவரையை** உருவாக்கப் போகிறீர்கள், இது விடுமுறை இடங்களை பரிந்துரைக்கும். இதனுடன் நீங்கள் கற்கப்போகும் விஷயங்கள்:

1. **Microsoft Agent Framework** பயன்படுத்தி Azure AI Foundry Agent சேவையுடன் இணைபு கட்டமைக்க.
2. முகவருக்கு ஒரு **கருவி** கொடுக்க — அது அழைக்கக்கூடிய ஒரு சாதாரண Python செயல்பாடு.
3. முகவரைக் இயக்கி அதன் பதிலைப் பரிசீலனை செய்ய.
4. முகவரின் பதிலை ஒவ்வொரு டோக்கனாக, ஒவ்வொன்றாக ஒளிர்ச்சி மூலம் பெற.


## அமைப்பு

இந்த நோட்டுபுக் இயக்குவதற்கு முன், நீங்கள் உறுதி செய்க:

1. **வெளியீட்டப்பட்ட உரையாடல் மாடல் (எ.கா. `gpt-4o-mini`) கொண்ட Azure AI Foundry திட்டம்** இருக்க வேண்டும்.
2. **Azure CLI மூலம் உள்நுழைந்திருப்பதை** — உங்கள் டெர்மினலில் `az login` இயக்குக.
3. **தேவைப்படும் சுற்றுச்சூழல் மாறிலிகள் அமைக்கப்பட்டிருக்க வேண்டும்:**
   - `AZURE_AI_PROJECT_ENDPOINT` — உங்கள் Azure AI Foundry திட்டத்தின் முடிவுலகம்.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — உங்கள் வெளியீட்டப்பட்ட மாடலின் பெயர்.

கீழே உள்ள செல்ல் நீங்கள் தேவையான Python தொகுப்புக்களை நிறுவுகிறது.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## உங்கள் முதல் முகவரியை உருவாக்குதல்

ஒரு முகவரிக்கு இரண்டு விஷயங்கள் தேவை:

- **அறிவுறுத்தல்கள்** அது *யார் என்று* மற்றும் *எப்படி நடந்து கொள்ள வேண்டும்* என்பதைக் கூறும் (ஒரு சிஸ்டம் ப்ராம்ட்).
- **கருவிகள்** — முகவர் தகவலை பெற அல்லது செயல்களை நிறைவேற்ற அழைக்கக் கூடிய `@tool` மூலம் அலங்கரிக்கப்பட்ட பைதான் செயல்பாடுகள்.

கீழே பிரபலமான விடுமுறை இடங்களின் பட்டியலை திருப்பி அளிக்கும் ஒரு எளிய கருவியை வரையறுக்கிறோம். பயனர் பயண பரிந்துரைகள் கேட்கும் போது முகவர் இந்த கருவியை பயன்படுத்தும்.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ஸ்ட்ரீமிங் பதில்கள்

மேலும் பரிமாற்றம் வாய்ந்த அனுபவத்திற்காக நீங்கள் முகவரியின் பதிலை **ஸ்ட்ரீம்** செய்யலாம். முழுமையான பதிலை காத்திருக்காமையுள்ளதின் பதிலாக, முகவர் உருவாக்கப்படும் போதே உரை துணுக்குகளை வழங்குகிறான். இது நேரடி வெளிப்பாட்டை காட்ட விரும்பும் உரையாடல் இடைமுகங்களில் மிகவும் பயனுள்ளதாக இருக்கும்.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் எப்படி என்பதை கற்றுக்கொண்டீர்கள்:

- **AzureAIProjectAgentProvider** மூலம் Azure AI Foundry Agent சேவையுடன் இணைக்கும் ஒரு வழங்குநரை உருவாக்கவும்.
- பிரதிநிதி உங்கள் Python செயல்பாடுகளை அழைக்க `@tool` அலங்கரிப்பை பயன்படுத்தி ஒரு கருவியை வரையறுக்கவும்.
- பயனர் செய்தியுடன் பிரதிநிதியை இயக்கி அதன் பதிலை அச்சிடவும்.
- நேரடி வெளியீட்டிற்கு பதில்களை ஸ்ட்ரீம் செய்யவும்.

அடுத்த பாடத்தில், நாங்கள் முகவருப்பு கட்டமைப்புகளை மேலும் ஆழமாக ஆராய்ந்து, முகவர்கள் மேலும் சக்திவாய்ந்த கருவிகள் மற்றும் பலநிலை தர்கவியல் திறன்களை எப்படி வழங்குவது என்பதை கற்போம்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**வெளிப்படுத்தல்**:  
இந்தக் கோப்புறையை AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) மூலம் மொழிபெயர்க்கப்பட்டுள்ளது. நாம் துல்லியத்திற்காக முயلعிறோம் என்றாலும், தானியங்கி மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கக்கூடும் என்பதை உணர்த்துகிறோம். இதன் ஒரிஜினல் மொழியில் உள்ள ஆவணம் அதிகாரப்பூர்வமான ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தனிப்பட்ட மக்களின் தொழில்முறை மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பை பயன்படுத்துவதில் ஏற்படக்கூடிய எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கங்களுக்கு நாங்கள் பொறுப்பானவர்கள் அல்ல.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
